In [4]:
import os
import json
from glob import glob
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [5]:
def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.

    Args:
        image_url (str): URL of the image to download
        save_path (str): Local path where to save the image (including filename)
        timeout (int): Request timeout in seconds

    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        with httpx.Client() as client:
            response = client.get(image_url, timeout=timeout)
            response.raise_for_status()  # Raise an exception for bad status codes

            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # Write the image to file
            with open(save_path, "wb") as file:
                file.write(response.content)

        # print(f"✓ Downloaded: {save_path}")
        return True

    except httpx.RequestError as e:
        print(f"✗ Failed to download {image_url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error saving image: {e}")
        return False


def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

In [11]:
# datafile = str(DATA_DIR / "german" / "politicians")
datafile = str(DATA_DIR / "vietnam" / "01102025_meds")
datapaths = glob(datafile + "/**.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/doctors_bvntw.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/phuongdong.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/benhvien175_doctors.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/benhvien103_doctors.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/fv_doctors.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/bv_daihoc.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/vietnamcuba.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/01102025_meds/namsaigon_doctors.json']

### Profiling

Normalize to this format

```
from dataclasses import dataclass

@dataclass
class Profile:
    name: str
    mainType: str # Job
    dateOfBirth: str
    homePlace: str  # where the person born from
    workPlace: str
    gender: str
    image_url: str
    profile_url: str
    other_info: dict

```

In [19]:
final_profiles = []

for path in datapaths:
    profiles = read_json(file_path=path)

    for profile in profiles:
        data = {}
        data["name"] = profile.get("name", "").title()
        data["mainType"] = profile.get("mainType", "")
        data["dateOfBirth"] = profile.get("dateOfBirth", "")
        data["homePlace"] = profile.get("homePlace", "")
        data["workPlace"] = profile.get("workPlace", "")
        data["gender"] = profile.get("gender", "")
        data["image_url"] = profile.get("image_url", "")
        data["profile_url"] = profile.get("profile_url", "")

        final_profiles.append(data)

len(final_profiles)

1161

In [20]:
write_json(
    data=final_profiles,
    file_path="../data/med_vn_profiles_12102025_001_before.json",
)

✓ Written JSON to: ../data/med_vn_profiles_12102025_001_before.json


True

In [21]:
"""
Only get profile that get image
"""

usable_profiles = []
save_dir = str(DATA_DIR / "images" / "med_vn_profiles")


for idx in tqdm(range(len(final_profiles))):
    try:
        profile = final_profiles[idx]
        image_url = profile["image_url"]
        img_name = profile["name"].replace(" ", "_")
        local_path = f"{save_dir}/{img_name}.jpg"
        profile["local_path"] = local_path
        is_success = download_image(image_url, local_path)
        if is_success:
            usable_profiles.append(profile)

    except Exception as e:
        print(e)
        continue

  0%|          | 0/1161 [00:00<?, ?it/s]

✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20364%20450'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.


  0%|          | 2/1161 [00:01<13:43,  1.41it/s]

✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20150%20150'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20364%20450'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.


  1%|▏         | 15/1161 [00:03<02:05,  9.12it/s]

...

 12%|█▏        | 138/1161 [00:31<01:10, 14.44it/s]

✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.


 13%|█▎        | 151/1161 [00:32<00:48, 20.76it/s]

✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20601'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.
✗ Failed to download data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20400%20600'%3E%3C/svg%3E: Request URL is missing an 'http://' or 'https://' protocol.


100%|██████████| 1161/1161 [02:51<00:00,  6.78it/s]


In [23]:
write_json(
    data=usable_profiles,
    file_path="../data/med_vn_profiles_12102025_001_after.json",
)

✓ Written JSON to: ../data/med_vn_profiles_12102025_001_after.json


True

In [22]:
len(usable_profiles)

1133